In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

## Loading data

In [ ]:
# Read all tables
customers = pd.read_csv('data/customers_final.csv')
engagements = pd.read_csv('data/engagements_final.csv')
marketing = pd.read_csv('data/marketing_final.csv')
transactions = pd.read_csv('data/transactions_final.csv')

In [ ]:
customers.head()

In [ ]:
customers.info()

In [ ]:
engagements.head()

In [ ]:
engagements.info()

In [ ]:
marketing.head()

In [ ]:
marketing.info()

In [ ]:
transactions.head()

In [ ]:
transactions.info()

## Converting date columns to datetime

In [ ]:
customers['join_date'] = pd.to_datetime(customers['join_date'])
customers['last_purchase_date'] = pd.to_datetime(customers['last_purchase_date'])
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])

## Filling missing values

In [ ]:
median_age = customers['age'].median()
mode_gender = customers['gender'].mode()[0]
customers['age'] = customers['age'].fillna(median_age)
customers['gender'] = customers['gender'].fillna(mode_gender)

## Data aggregation and consolidation into a single dataset

In [ ]:
# Creating aggregate dataframes
marketing_agg = marketing[marketing['response'] == 'Yes'].groupby('customer_id')['campaign_id'].count().to_frame()
transactions_agg = transactions.groupby('customer_id').aggregate(
    {'transaction_id': 'count', 'transaction_amount': 'sum'})

# Setting indices for joining
customers.set_index('customer_id', inplace=True)
engagements.set_index('customer_id', inplace=True)

# Joining data
joint_data = customers.join(engagements).join(transactions_agg).join(marketing_agg)
joint_data['campaign_id'] = joint_data['campaign_id'].fillna(0)

joint_data.head(10)

# Feature engineering

In [ ]:
# Calculating customer tenure in the company in days
joint_data['customer_tenure'] = (pd.Timestamp('today') - pd.to_datetime(joint_data['join_date'])).dt.days

# Calculating the average time between purchases for each customer
transactions['days_between_purchases'] = transactions.sort_values(
    ['customer_id', 'transaction_date']).groupby('customer_id')['transaction_date'].diff().dt.days
avg_days_between_purchases = transactions.groupby('customer_id')['days_between_purchases'].mean()
joint_data = joint_data.join(avg_days_between_purchases.rename('avg_days_between_purchases'), on='customer_id')

# Handling missing values in 'avg_days_between_purchases' by filling with median
median_days = joint_data['avg_days_between_purchases'].median()
joint_data['avg_days_between_purchases'] = joint_data['avg_days_between_purchases'].fillna(median_days)

# Count the number of marketing campaigns each customer has participated in
num_marketing_campaigns = marketing.groupby('customer_id').size()
joint_data['num_marketing_campaigns'] = joint_data.index.map(num_marketing_campaigns).fillna(0)

# Calculating the number of days since the last purchase
joint_data['days_since_last_purchase'] = (pd.Timestamp('today') - joint_data['last_purchase_date']).dt.days

# Calculate the response rate to marketing campaigns for each customer
marketing['response_binary'] = marketing['response'].apply(lambda x: 1 if x == 'Yes' else 0)
marketing_response_rate = marketing.groupby('customer_id')['response_binary'].mean()
joint_data['marketing_response_rate'] = joint_data.index.map(marketing_response_rate).fillna(0)

joint_data.head()

## LTV calculation

In [ ]:
# Calculate the average revenue per purchase
joint_data['avg_revenue_per_purchase'] = joint_data['transaction_amount'] / joint_data['transaction_id']

# Calculate the purchase frequency
joint_data['purchase_frequency'] = joint_data['transaction_id'] / joint_data['customer_tenure']

# Calculate LTV using average revenue per purchase and purchase frequency
joint_data['LTV'] = joint_data['avg_revenue_per_purchase'] * joint_data['purchase_frequency'] * joint_data['customer_tenure']

# Display the updated DataFrame with the LTV column
joint_data.head()

## Coding of categorical variables

In [ ]:
# Apply OneHot encoding to 'gender'
onehot_encoder = OneHotEncoder(sparse_output=False)
gender_encoded = onehot_encoder.fit_transform(joint_data[['gender']])

# Convert to DataFrame
gender_encoded_df = pd.DataFrame(gender_encoded, columns=onehot_encoder.get_feature_names_out(['gender']), index=joint_data.index)

# Join the encoded gender data back to the main DataFrame
joint_data_encoded = joint_data.join(gender_encoded_df)

# Drop unneeded columns
columns_to_drop = ['location', 'gender', 'gender_Female', 'join_date', 'last_purchase_date']
joint_data_encoded.drop(columns_to_drop, axis=1, inplace=True)

# Display the head of the updated DataFrame
joint_data_encoded.head()

## Calculate the correlation matrix

In [ ]:
correlation_matrix = joint_data_encoded.corr()

# Set up the matplotlib figure
plt.figure(figsize=(12, 10))

# Draw the heatmap
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True)

# Display the heatmap
plt.show()

## Scaling of numeric features using MinMaxScaler

In [ ]:
scaler = MinMaxScaler()
scale_columns = ['age', 'number_of_site_visits', 'number_of_emails_opened', 'number_of_clicks',
                 'transaction_amount', 'LTV', 'customer_tenure', 'avg_days_between_purchases',
                 'num_marketing_campaigns', 'marketing_response_rate', 'days_since_last_purchase', 'avg_revenue_per_purchase']

# Apply scaling to selected columns
joint_data_encoded[scale_columns] = scaler.fit_transform(joint_data_encoded[scale_columns])

joint_data_encoded.head()

## Define the target variable and features

In [ ]:
X = joint_data_encoded.drop(columns=['LTV', 'transaction_amount', 'transaction_id'])
y = joint_data_encoded['LTV']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Model initialization and training

In [ ]:
# Initialize models
linear_reg = LinearRegression()
rfr = RandomForestRegressor()
xgb = XGBRegressor()

# Train models
linear_reg.fit(X_train, y_train)
rfr.fit(X_train, y_train)
xgb.fit(X_train, y_train)

## # Model prediction and evaluation

In [ ]:
# Prediction and evaluation of models
models = {
    'Linear Regression': linear_reg,
    'Random Forest Regressor': rfr,
    'XGBoost Regressor': xgb
}

results = {}

for model_name, model in models.items():
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results[model_name] = {'MSE': mse, 'R2': r2}

# Formation of results
results_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest Regression', 'XGBoost Regression'],
    'MSE (Mean Square Error)': [
        f'{results["Linear Regression"]["MSE"]:.8f}',
        f'{results["Random Forest Regressor"]["MSE"]:.8f}',
        f'{results["XGBoost Regressor"]["MSE"]:.8f}'
    ],
    'R2 (Coefficient of Determination)': [
        f'{results["Linear Regression"]["R2"]:.4f}',
        f'{results["Random Forest Regressor"]["R2"]:.4f}',
        f'{results["XGBoost Regressor"]["R2"]:.4f}'
    ]
})

# Display results
results_df

## Displaying the importance of features

In [ ]:
# Display feature importances for each model
def display_feature_importances(model, model_name):
    if hasattr(model, 'coef_'):
        importance = model.coef_
    else:
        importance = model.feature_importances_
    
    features = X_train.columns
    importance_df = pd.DataFrame({'Feature': features, 'Importance': importance}).sort_values(by='Importance', ascending=False)
    
    print(f'Importance of features for the model {model_name}')
    print(importance_df)

# Display feature importances for each model
for model_name, model in models.items():
    display_feature_importances(model, model_name)